# 07 — Trees and ensembles: one split at a time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/007-trees-and-ensembles.ipynb)

A decision tree is a stack of yes/no questions, and every question is chosen to make the two groups it creates as pure as possible. This notebook measures purity with the Gini index, finds the best single split by trying every threshold, and shows when splitting cannot help at all — which is exactly why forests average many weak, barely-helpful splits.

## 1. The numbers: twelve rows, one feature

The feature is a score from 1 to 12 and the label is yes or no. Six rows are yes and six are no, so the starting group is as mixed as it can be — a coin flip. A split is only useful if the two halves are less mixed than the whole.

In [ ]:
import numpy as np                                        # the array library
x = np.arange(1, 13, dtype=float)                         # one feature, twelve ordered rows
y = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1], dtype=float)   # the label: six zeros, six ones
print("x:", x)                                            # the feature values
print("y:", y)                                            # the labels
print("zeros:", int(len(y) - y.sum()), " ones:", int(y.sum()))   # the class counts
print("share of ones:", round(y.mean(), 3))               # the base rate: a coin flip
p = y.mean()                                              # fraction of ones in the whole group
print("Gini of the whole group:", round(1 - p ** 2 - (1 - p) ** 2, 3))   # 0.5 means perfectly mixed

**Line by line**

- `import numpy as np` — loads NumPy.
- `x = np.arange(1, 13, dtype=float)` — the single feature, twelve rows in order.
- `y = ...` — the label, 0 or 1, six of each.
- `print("x:" ...)` — the feature values.
- `print("y:" ...)` — the labels, in row order.
- `print("zeros" ...)` — counts each class.
- `print("share of ones" ...)` — the base rate, 0.5.
- `p = y.mean()` — the share of ones is the only number Gini needs.
- `print("Gini ...")` — `1 - p^2 - (1-p)^2`, which is 0.5 for a 50/50 group.

## 2. Step by step: try every threshold, keep the purest

There is no formula for the best split on one feature — you try every gap between neighbouring rows and score the result. The score is the size-weighted impurity of the two halves, and the smallest score wins.

In [ ]:
def gini(labels):                                         # impurity of one group
    if len(labels) == 0:                                  # guard against an empty side
        return 0.0                                        # an empty group adds nothing
    q = labels.mean()                                     # share of ones inside this group
    return 1 - q ** 2 - (1 - q) ** 2                      # 0 = pure, 0.5 = a coin flip

cuts = x[:-1] + 0.5                                      # cut halfway between neighbouring rows
child = []                                               # weighted child impurity per cut
for t in cuts:                                           # one iteration per candidate split
    left = y[x <= t]                                     # labels sent left
    right = y[x > t]                                     # labels sent right
    score = (len(left) * gini(left) + len(right) * gini(right)) / len(y)   # size-weighted impurity
    child.append(score)                                  # keep the score
    print(f"t={t:5.1f} left={len(left):2d} right={len(right):2d} impurity={score:.3f}")   # trace every split
best = int(np.argmin(child))                             # the purest pair of groups
print("best cut:", cuts[best], " impurity:", round(child[best], 3))   # the winner
print("gain over no split:", round(gini(y) - child[best], 3))         # how much purity it bought

**Line by line**

- `def gini(labels)` — a function that scores one group.
- `if len(labels) == 0` — an empty side would divide by zero.
- `return 0.0` — treat an empty group as harmless.
- `q = labels.mean()` — the share of ones in the group.
- `return 1 - q ** 2 - (1 - q) ** 2` — the Gini index.
- `cuts = x[:-1] + 0.5` — every gap between neighbouring feature values.
- `child = []` — collects one score per candidate.
- `for t in cuts` — brute force over all eleven thresholds.
- `left = y[x <= t]` — the labels on the left of the cut.
- `right = y[x > t]` — the labels on the right.
- `score = ...` — weighted average of the two child impurities, weighted by size.
- `child.append(score)` — record it in order.
- `print(f"t={t:5.1f} ...")` — one line per threshold, so the search is visible.
- `best = int(np.argmin(child))` — index of the lowest score.
- `print("best cut" ...)` — the winning threshold and its impurity.
- `print("gain" ...)` — how much purity the split bought.

## 3. Plot: impurity against the threshold

Plot the child impurity for every candidate cut. The curve dips at the threshold that separates the zeros from the ones; the dashed line is the impurity you get by not splitting at all, so the gap between the curve and the line is the gain.

In [ ]:
import matplotlib.pyplot as plt                          # the plotting library
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(cuts, child, "o-", color="tab:blue", label="impurity after the split")   # the search curve
plt.axhline(gini(y), color="grey", linestyle="--", label="no split")              # the baseline
plt.scatter([cuts[best]], [child[best]], color="tab:red", zorder=3, label="best cut")   # the winner
plt.xlabel("threshold on the feature")                   # x-axis label
plt.ylabel("size-weighted child impurity")               # y-axis label
plt.legend(fontsize=8)                                   # name the three elements
plt.title("The dip is the best question the tree can ask")   # the takeaway
plt.show()                                               # render

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library.
- `plt.figure(...)` — a wide, short figure.
- `plt.plot(cuts, child, "o-")` — impurity at every candidate threshold.
- `plt.axhline(gini(y), ...)` — the no-split baseline, drawn as a dashed line.
- `plt.scatter(...)` — marks the winning threshold in red.
- `plt.xlabel(...)` — the threshold being tried.
- `plt.ylabel(...)` — the impurity of the two groups it creates.
- `plt.legend(...)` — a small legend so all three marks are named.
- `plt.title(...)` — says what the dip means.
- `plt.show()` — renders the figure.

## 4. Interactive: require a minimum group size

A split that leaves one row on a side is usually noise, so real trees refuse splits with small children. Drag the minimum group size up: legal thresholds disappear, the dip flattens, and past six rows per side no split is allowed at all.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function

def curve_for(min_leaf=1):                                # impurity curve for one minimum group size
    out = []                                             # one entry per threshold
    for t in cuts:                                       # the same candidates as section 2
        left = y[x <= t]                                 # labels sent left
        right = y[x > t]                                 # labels sent right
        ok = len(left) >= min_leaf and len(right) >= min_leaf   # is this split allowed?
        out.append((len(left) * gini(left) + len(right) * gini(right)) / len(y) if ok else np.nan)   # blank when not
    return np.array(out)                                 # NaN marks a forbidden threshold

def show(min_leaf=1):                                     # min_leaf comes from the slider
    c = curve_for(min_leaf)                              # recompute the curve for this rule
    plt.figure(figsize=(6, 3))                           # a fresh figure per drag
    plt.plot(cuts, c, "o-", color="tab:blue")            # legal thresholds only, gaps where forbidden
    plt.axhline(gini(y), color="grey", linestyle="--")   # the no-split baseline
    if np.all(np.isnan(c)):                              # nothing left to try
        plt.title(f"min group {min_leaf}: no legal split, predict {y.mean():.2f}")   # fall back to the majority
    else:                                                # some splits are still allowed
        j = int(np.nanargmin(c))                          # best legal threshold
        plt.title(f"min group {min_leaf}: best t={cuts[j]:.1f}, gain={gini(y) - c[j]:.3f}")   # live readout
    plt.xlabel("threshold on the feature")               # x-axis label
    plt.ylabel("size-weighted child impurity")           # y-axis label
    plt.ylim(0, 0.55)                                    # fixed axes so frames do not jump
    plt.show()                                           # draw the frame

interact(show, min_leaf=(1, 7, 1))                       # attach the slider

**Line by line**

- `from ipywidgets import interact` — the widget helper.
- `def curve_for(min_leaf=1)` — builds one curve per minimum group size.
- `out = []` — collects eleven entries.
- `for t in cuts` — same brute force as section 2.
- `left = ...` — the labels on the left of this cut.
- `right = ...` — the labels on the right.
- `ok = ...` — a split is legal only if both sides are big enough.
- `out.append(... if ok else np.nan)` — illegal thresholds become gaps in the line.
- `return np.array(out)` — an array with `nan` where splits are banned.
- `def show(min_leaf=1)` — redraws a frame per slider value.
- `c = curve_for(...)` — recompute under the current rule.
- `plt.figure(...)` — a fresh figure per drag.
- `plt.plot(cuts, c, "o-")` — the curve, broken at forbidden thresholds.
- `plt.axhline(...)` — the baseline to beat.
- `if np.all(np.isnan(c))` — the no-split case.
- `plt.title(f"...")` — reports that the tree must stop and use the majority label.
- `else` — the case where at least one split is legal.
- `j = int(np.nanargmin(c))` — best legal threshold, ignoring the gaps.
- `plt.title(f"...")` — live readout of threshold and gain.
- `plt.xlabel(...)` — the threshold.
- `plt.ylabel(...)` — the resulting impurity.
- `plt.ylim(0, 0.55)` — fixed axes so frames do not jump.
- `plt.show()` — draws the frame.
- `interact(show, min_leaf=(1, 7, 1))` — slider from 1 to 7 rows per side.

## 5. When it breaks: a split that buys nothing

Shuffle the labels so the feature says nothing about the answer. Every threshold that keeps a couple of rows on each side then leaves the impurity almost exactly where it started: the best gain falls to 0.02, which is chance on twelve rows, not signal. Averaging many such splits — a forest — cannot invent signal that is not there.

In [ ]:
y_noise = np.tile([0.0, 1.0], 6)                         # labels that alternate regardless of x
curve_n = []                                             # impurity per threshold for the useless feature
for t in cuts:                                           # the same eleven candidates
    left = y_noise[x <= t]                               # left group
    right = y_noise[x > t]                               # right group
    curve_n.append((len(left) * gini(left) + len(right) * gini(right)) / len(y))   # weighted impurity
print("parent impurity:", round(gini(y_noise), 3))       # the starting point
print("best child impurity:", round(min(curve_n), 3))    # the best any threshold manages
print("best gain:", round(gini(y_noise) - min(curve_n), 3))   # tiny, and only from a group of one row
allowed = [c for t, c in zip(cuts, curve_n) if (y_noise[x <= t]).size >= 2 and (y_noise[x > t]).size >= 2]   # splits with two rows per side
print("best gain with two rows per side:", round(gini(y_noise) - min(allowed), 3))   # collapses to 0.019: chance
print("base rate:", round(y_noise.mean(), 3))            # all a forest can fall back on
stumps = []                                              # one full prediction per stump
rng = np.random.default_rng(3)                           # a fixed seed for the random cuts
for s in range(25):                                      # twenty-five one-question trees
    t = rng.uniform(cuts[0], cuts[-1])                   # each stump gets a random threshold
    pred = np.where(x <= t, y_noise[x <= t].mean(), y_noise[x > t].mean())   # left rate, else right rate
    stumps.append(pred)                                  # store its twelve predictions
print("average of 25 stump predictions:", round(float(np.mean(stumps)), 3))   # lands back on the base rate
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(cuts, curve_n, "o-", color="tab:red", label="useless feature")   # a nearly flat curve
plt.axhline(gini(y_noise), color="grey", linestyle="--", label="no split")   # the same height
plt.xlabel("threshold on the feature")                   # x-axis label
plt.ylabel("size-weighted child impurity")               # y-axis label
plt.legend(fontsize=8)                                   # name both lines
plt.title("A useless feature barely dents the baseline")   # the takeaway
plt.show()                                               # render

**Line by line**

- `y_noise = np.tile([0.0, 1.0], 6)` — labels that alternate, independent of the feature.
- `curve_n = []` — one impurity per threshold.
- `for t in cuts` — try every candidate anyway.
- `left = ...` — the left group's labels.
- `right = ...` — the right group's labels.
- `curve_n.append(...)` — the same size-weighted score as before.
- `print("parent impurity" ...)` — where the group starts, 0.5.
- `print("best child impurity" ...)` — the lowest score the search finds.
- `print("best gain" ...)` — 0.045, bought only by isolating a single row.
- `allowed = [...]` — keeps only splits with at least two rows on each side.
- `print("best gain with two rows per side" ...)` — collapses to 0.019, pure chance.
- `print("base rate" ...)` — the only prediction left.
- `stumps = []` — collects one full prediction vector per stump.
- `rng = np.random.default_rng(3)` — a fixed seed, so the forest is reproducible.
- `for s in range(25)` — twenty-five shallow trees.
- `t = rng.uniform(...)` — a random threshold for each stump.
- `pred = np.where(...)` — the stump predicts its left group's rate, or its right group's.
- `stumps.append(pred)` — keep its twelve predictions.
- `print("average of 25 ...")` — the ensemble lands on 0.5, the base rate.
- `plt.figure(...)` — a wide, short figure.
- `plt.plot(cuts, curve_n, ...)` — a curve that only grazes the baseline.
- `plt.axhline(...)` — the baseline it fails to beat.
- `plt.xlabel(...)` — the threshold.
- `plt.ylabel(...)` — the resulting impurity.
- `plt.legend(...)` — names the two lines.
- `plt.title(...)` — states the lesson.
- `plt.show()` — renders the figure.

## Your turn

Change one label in `y` — say `y[3] = 1` — and rerun section 2: the best cut and the gain both move. Then set `min_leaf` to 4 in section 4 and explain why the tree now prefers a worse threshold, and what a forest gains by averaging many such constrained trees.